# Combined versus separate datasets: stability and calibration

This experiment addresses two decisions for the current Inception-style CNN candidate:

1. Should Voisard and Felius be pooled for training, or should separate source-specific models be maintained?
2. Are the pooled results stable across random seeds and reasonably calibrated?

Pooled training uses both datasets but keeps participants—not windows—as the split unit. Normalization statistics are calculated separately inside each training fold. The existing separate-source cross-dataset results are loaded for comparison.

In [1]:
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import balanced_accuracy_score, brier_score_loss, f1_score, log_loss, roc_auc_score
from torch import nn
from torch.utils.data import DataLoader, Dataset

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
PROCESSED = PROJECT_ROOT / 'data' / 'processed'
INTERIM = PROJECT_ROOT / 'data' / 'interim'
MAG_PATH = PROCESSED / 'validated_acceleration_magnitude_windows_float32.npy'
METADATA_PATH = PROCESSED / 'validated_window_metadata.csv'
SPLITS_PATH = INTERIM / 'participant_splits.csv'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.set_num_threads(4)
EPOCHS = 4
BATCH_SIZE = 128
metadata = pd.read_csv(METADATA_PATH)
metadata['label_binary'] = metadata['label'].map({'healthy': 0, 'stroke': 1}).astype(int)
splits = pd.read_csv(SPLITS_PATH)
magnitude_windows = np.load(MAG_PATH, mmap_mode='r')
print('Device:', DEVICE)
print('Windows:', magnitude_windows.shape)
print(metadata.groupby(['dataset_id', 'label']).participant_key.nunique())

Device: cuda
Windows: (18511, 500, 3)
dataset_id    label  
felius_2024   healthy     34
              stroke     129
voisard_2025  healthy     72
              stroke      49
Name: participant_key, dtype: int64


In [2]:
def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def fold_indices(fold):
    role_map = splits[splits['fold'].eq(fold)].set_index('participant_key')['role']
    roles = metadata['participant_key'].map(role_map)
    return np.flatnonzero(roles.eq('training').to_numpy()), np.flatnonzero(roles.eq('validation').to_numpy())

def normalization(indices):
    total = np.zeros(3, dtype='float64'); total_sq = np.zeros(3, dtype='float64'); count = 0
    for start in range(0, len(indices), 512):
        batch = np.asarray(magnitude_windows[indices[start:start + 512]], dtype='float32')
        total += batch.sum(axis=(0, 1)); total_sq += np.square(batch).sum(axis=(0, 1)); count += batch.shape[0] * batch.shape[1]
    mean = total / count; std = np.sqrt(np.maximum(total_sq / count - mean ** 2, 1e-8))
    return mean.astype('float32'), std.astype('float32')

def sample_weights(indices):
    frame = metadata.iloc[indices]
    pcounts = frame.groupby('participant_key').size(); ccounts = frame.groupby('label_binary').size()
    weights = frame['participant_key'].map(1.0 / pcounts).to_numpy()
    weights *= frame['label_binary'].map(len(indices) / (2.0 * ccounts)).to_numpy()
    return (weights / weights.mean()).astype('float32')

class GaitDataset(Dataset):
    def __init__(self, indices, mean, std, weights=None):
        self.indices = np.asarray(indices, dtype='int64'); self.mean = mean.reshape(1, 3); self.std = std.reshape(1, 3)
        self.weights = np.ones(len(self.indices), dtype='float32') if weights is None else weights
    def __len__(self): return len(self.indices)
    def __getitem__(self, item):
        index = int(self.indices[item])
        signal = ((np.asarray(magnitude_windows[index], dtype='float32') - self.mean) / self.std).T.copy()
        return torch.from_numpy(signal), torch.tensor(float(metadata.iloc[index]['label_binary'])), torch.tensor(float(self.weights[item])), torch.tensor(index)

In [3]:
class InceptionBlock(nn.Module):
    def __init__(self, in_channels, out_channels=16):
        super().__init__(); bottleneck = min(32, in_channels)
        self.bottleneck = nn.Conv1d(in_channels, bottleneck, 1, bias=False)
        self.branches = nn.ModuleList([nn.Conv1d(bottleneck, out_channels, 7, padding=3, bias=False), nn.Conv1d(bottleneck, out_channels, 15, padding=7, bias=False), nn.Conv1d(bottleneck, out_channels, 25, padding=12, bias=False)])
        self.pool_branch = nn.Conv1d(in_channels, out_channels, 1, bias=False)
        self.bn = nn.BatchNorm1d(out_channels * 4)
        self.residual = nn.Conv1d(in_channels, out_channels * 4, 1, bias=False) if in_channels != out_channels * 4 else nn.Identity()
    def forward(self, x):
        z = self.bottleneck(x); branches = [branch(z) for branch in self.branches]
        branches.append(self.pool_branch(nn.functional.max_pool1d(x, 3, stride=1, padding=1)))
        return nn.functional.gelu(self.bn(torch.cat(branches, dim=1)) + self.residual(x))

class InceptionCNN(nn.Module):
    def __init__(self):
        super().__init__(); self.features = nn.Sequential(InceptionBlock(3), nn.MaxPool1d(2), InceptionBlock(64), nn.AdaptiveAvgPool1d(1)); self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(0.30), nn.Linear(64, 1))
    def forward(self, x): return self.classifier(self.features(x)).squeeze(1)

def predict(model, loader):
    model.eval(); rows = []
    with torch.no_grad():
        for signals, _, _, indices in loader:
            probabilities = torch.sigmoid(model(signals.to(DEVICE))).detach().cpu().numpy()
            rows.extend(zip(indices.numpy(), probabilities))
    return np.array([p for _, p in rows], dtype='float32')

def participant_frame(indices, probabilities):
    frame = metadata.iloc[np.asarray(indices)].copy(); frame['probability'] = probabilities
    return frame.groupby(['participant_key', 'dataset_id', 'label_binary'], as_index=False)['probability'].mean()

def metric_row(frame):
    y = frame['label_binary'].to_numpy(); p = frame['probability'].to_numpy(); pred = (p >= 0.5).astype(int)
    return {'participants': len(frame), 'balanced_accuracy': balanced_accuracy_score(y, pred), 'roc_auc': roc_auc_score(y, p), 'f1': f1_score(y, pred)}

In [4]:
def train_pooled_fold(fold, seed):
    set_seed(seed); train_indices, validation_indices = fold_indices(fold); mean, std = normalization(train_indices)
    train_loader = DataLoader(GaitDataset(train_indices, mean, std, sample_weights(train_indices)), batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    validation_loader = DataLoader(GaitDataset(validation_indices, mean, std), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    model = InceptionCNN().to(DEVICE); optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    best_auc = -np.inf; best_state = None; patience = 2
    for epoch in range(1, EPOCHS + 1):
        model.train()
        for signals, labels, weights, _ in train_loader:
            optimizer.zero_grad(); logits = model(signals.to(DEVICE))
            loss = (nn.functional.binary_cross_entropy_with_logits(logits, labels.to(DEVICE), reduction='none') * weights.to(DEVICE)).mean(); loss.backward(); optimizer.step()
        val_probabilities = predict(model, validation_loader); val_frame = participant_frame(validation_indices, val_probabilities); val_metrics = metric_row(val_frame)
        if val_metrics['roc_auc'] > best_auc:
            best_auc = val_metrics['roc_auc']; best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}; patience = 2
        else:
            patience -= 1
            if patience == 0: break
    model.load_state_dict(best_state); val_probabilities = predict(model, validation_loader); val_frame = participant_frame(validation_indices, val_probabilities)
    val_frame['fold'] = fold; val_frame['seed'] = seed; val_frame['model'] = 'pooled_inception'
    return val_frame, {'fold': fold, 'seed': seed, **metric_row(val_frame)}

In [5]:
pooled_frames = []; pooled_fold_results = []
for fold in range(5):
    frame, metrics = train_pooled_fold(fold, 42); pooled_frames.append(frame); pooled_fold_results.append(metrics)
pooled_oof = pd.concat(pooled_frames, ignore_index=True); pooled_fold_results = pd.DataFrame(pooled_fold_results)
pooled_oof_metrics = metric_row(pooled_oof)
print('Pooled OOF:', {k: round(v, 3) for k, v in pooled_oof_metrics.items()})
print('Pooled OOF by source:')
print(pooled_oof.groupby('dataset_id').apply(metric_row).to_string())

Pooled OOF: {'participants': 284, 'balanced_accuracy': 0.857, 'roc_auc': 0.957, 'f1': 0.854}
Pooled OOF by source:
dataset_id
felius_2024     {'participants': 163, 'balanced_accuracy': 0.8...
voisard_2025    {'participants': 121, 'balanced_accuracy': 0.8...


C:\Users\frank\AppData\Local\Temp\ipykernel_16288\2941151909.py:8: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  print(pooled_oof.groupby('dataset_id').apply(metric_row).to_string())


In [6]:
seed_rows = []
for seed in [42, 52, 62, 72, 82]:
    frame, metrics = train_pooled_fold(0, seed); metrics['model'] = 'pooled_inception_seed_stability'; seed_rows.append(metrics)
seed_results = pd.DataFrame(seed_rows)
print(seed_results.round(3).to_string(index=False))

 fold  seed  participants  balanced_accuracy  roc_auc    f1                           model
    0    42            57              0.817    0.929 0.836 pooled_inception_seed_stability
    0    52            57              0.772    0.942 0.754 pooled_inception_seed_stability
    0    62            57              0.827    0.927 0.831 pooled_inception_seed_stability
    0    72            57              0.794    0.931 0.824 pooled_inception_seed_stability
    0    82            57              0.845    0.939 0.870 pooled_inception_seed_stability


In [7]:
def expected_calibration_error(frame, bins=10):
    p = frame['probability'].to_numpy(); y = frame['label_binary'].to_numpy(); edges = np.linspace(0, 1, bins + 1); ece = 0.0
    for lower, upper in zip(edges[:-1], edges[1:]):
        mask = (p >= lower) & (p < upper if upper < 1 else p <= upper)
        if mask.any(): ece += mask.mean() * abs(p[mask].mean() - y[mask].mean())
    return float(ece)

calibration_rows = []
for source, frame in [('pooled_all', pooled_oof), *pooled_oof.groupby('dataset_id')]:
    calibration_rows.append({'scope': source, 'participants': len(frame), 'brier': brier_score_loss(frame['label_binary'], frame['probability']), 'log_loss': log_loss(frame['label_binary'], np.clip(frame['probability'], 1e-6, 1 - 1e-6)), 'ece_10bin': expected_calibration_error(frame)})
calibration = pd.DataFrame(calibration_rows)
separate = pd.read_csv(PROCESSED / 'cross_dataset_architecture_results.csv')
separate = separate[separate['model'].eq('inception_cnn_gpu')].copy()
separate['strategy'] = 'separate_source_model'
pooled_source = pooled_oof.groupby('dataset_id').apply(metric_row).apply(pd.Series).reset_index()
pooled_source['strategy'] = 'pooled_model'
pooled_source['model'] = 'pooled_inception'
pooled_source = pooled_source.rename(columns={'dataset_id': 'test_dataset'})
comparison = pd.concat([separate[['strategy', 'test_dataset', 'balanced_accuracy', 'roc_auc', 'f1']], pooled_source[['strategy', 'test_dataset', 'balanced_accuracy', 'roc_auc', 'f1']]], ignore_index=True)
print('Calibration:'); print(calibration.round(3).to_string(index=False))
print('Combined versus separate:'); print(comparison.round(3).to_string(index=False))
pooled_oof.to_csv(PROCESSED / 'pooled_inception_oof_predictions.csv', index=False)
pooled_fold_results.to_csv(PROCESSED / 'pooled_inception_fold_results.csv', index=False)
seed_results.to_csv(PROCESSED / 'pooled_inception_seed_stability.csv', index=False)
calibration.to_csv(PROCESSED / 'pooled_inception_calibration.csv', index=False)
comparison.to_csv(PROCESSED / 'combined_vs_separate_results.csv', index=False)

Calibration:
       scope  participants  brier  log_loss  ece_10bin
  pooled_all           284  0.120     0.372      0.182
 felius_2024           163  0.136     0.421      0.211
voisard_2025           121  0.099     0.306      0.142
Combined versus separate:
             strategy test_dataset  balanced_accuracy  roc_auc    f1
separate_source_model  felius_2024              0.711    0.876 0.642
separate_source_model voisard_2025              0.737    0.790 0.711
         pooled_model  felius_2024              0.830    0.916 0.874
         pooled_model voisard_2025              0.830    0.976 0.795


C:\Users\frank\AppData\Local\Temp\ipykernel_16288\560457582.py:15: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  pooled_source = pooled_oof.groupby('dataset_id').apply(metric_row).apply(pd.Series).reset_index()


## Decision rule

Prefer pooled training only if it improves both source-specific validation results without creating severe calibration or seed instability. If pooled training is strong on average but weak for one dataset, retain source-specific reporting and investigate domain adaptation or protocol harmonization before deployment claims.